In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schema names are loaded from 00_project_config.
# Reusable validation, cleaning, deduplication, and writing helpers are loaded
# from 01_spark_utils.

BRONZE_TABLE = f"{BRONZE_SCHEMA}.api_venues_data"
SILVER_TABLE = f"{SILVER_SCHEMA}.venues_data"

# One Silver row represents one API-Football venue.
BUSINESS_KEY = [
    "venue_id",
]

REQUIRED_BRONZE_COLUMNS = [
    "id",
    "name",
    "address",
    "city",
    "country",
    "capacity",
    "surface",
    "image",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

REQUIRED_SILVER_COLUMNS = [
    "venue_id",
    "venue_name",
]


# -----------------------------------------------------------------------------
# Validate Bronze input
# -----------------------------------------------------------------------------
# A missing Bronze table is treated as a pipeline failure because continuing
# could leave an existing Silver table stale or incomplete.

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )


# -----------------------------------------------------------------------------
# Read Bronze data
# -----------------------------------------------------------------------------
# The API-Football venue Bronze table is already relatively flat, so Silver
# mainly performs:
#
#   - string cleaning
#   - explicit datatype casting
#   - clearer analytical naming
#   - deterministic deduplication
#   - data-quality validation
#
# Bronze lineage metadata is retained for traceability.

bronze_df = spark.table(BRONZE_TABLE)

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Select, clean, and standardize Silver columns
# -----------------------------------------------------------------------------
# venue_id is intentionally stored as LONG.
#
# This aligns it with:
#   team_data.venue_id
#
# and removes the previous datatype inconsistency where venues_data.venue_id
# was STRING while team_data.venue_id was LONG.
#
# Venue columns receive explicit venue_* names so their meaning remains clear
# when joined with team, competition, or future Gold datasets.

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Venue identity
        # ---------------------------------------------------------------------

        F.col("id")
        .cast("long")
        .alias("venue_id"),

        clean_string("name")
        .alias("venue_name"),


        # ---------------------------------------------------------------------
        # Location
        # ---------------------------------------------------------------------

        clean_string("address")
        .alias("venue_address"),

        clean_string("city")
        .alias("venue_city"),

        clean_string("country")
        .alias("venue_country"),


        # ---------------------------------------------------------------------
        # Venue attributes
        # ---------------------------------------------------------------------

        F.col("capacity")
        .cast("int")
        .alias("venue_capacity"),

        clean_string("surface")
        .alias("venue_surface"),

        clean_string("image")
        .alias("venue_image_url"),


        # ---------------------------------------------------------------------
        # Bronze lineage metadata
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Future entity standardization placeholders
# -----------------------------------------------------------------------------
# The next project phase will standardize entity names and attributes across
# sources.
#
# Important candidates from this dataset include:
#
#   venue_name
#   venue_address
#   venue_city
#   venue_country
#
# Venue standardization will be especially important because the same stadium
# can potentially appear with different naming conventions across sources.
#
# Example future model:
#
#   venue_name_source
#   venue_name_standardized
#   venue_key
#
#   city_name_standardized
#   country_name_standardized
#
# The preferred approach will be to preserve source-derived values while adding
# canonical platform identifiers / standardized values.
#
# No physical placeholder columns are added yet because NULL-only or duplicate
# columns would add unnecessary schema noise before the matching logic exists.


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# Every usable venue record must have a venue_id.
#
# Validation happens before deduplication so records with NULL identifiers
# cannot be grouped together and accidentally reduced to one arbitrary row.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------
# Bronze currently uses append mode. Re-running the Bronze load against the
# same raw source can therefore create multiple versions of a venue.
#
# For each venue_id, retain the record from the latest Bronze ingestion.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# venue_id and venue_name are required for a usable venue entity.
#
# The helper also returns the final row count, avoiding a separate count()
# action.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"venue records into {SILVER_TABLE}."
)